# 🌍 Phase 2: Exploratory Data Analysis (EDA)

Welcome to the **Exploration Phase**.
In this notebook, we transform the rigorously cleaned data into a narrative, discovering patterns, anomalies, and insights.

**Key Goals:**
1.  **Univariate Analysis**: Understand the distribution of scores, popularity, and episodes (Normality vs Power Laws).
2.  **Bivariate Analysis**: Uncover correlations (Does popularity equal quality?).
3.  **Outlier Detection**: Categorize anomalies by distinguishing between "Masterpieces" (extreme statistical rarities where $Z > 3\sigma$) and "Infinite Series" (standard outliers identified via the Interquartile Range method).
4.  **Controversy Metric**: Identify anime with high voting variance (Love/Hate).
5.  **Geospatial**: Quick look at user demographics.

## 0. Environment Setup
Configuring pandas display options and loading project modules.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path
import pandas as pd
pd.set_option('display.max_columns', 200)
import seaborn as sns
import matplotlib.pyplot as plt

# 🔧 Dynamic Path Setup (Fixes ModuleNotFoundError)
# We need to access 'solution' package from standard path
current_dir = Path(os.getcwd())
project_root = current_dir.parent.parent # solution/notebooks -> solution -> root
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# 📦 Import Project Modules
from nb_02_exploration import (
    load_data,
    calculate_controversy_metric,
    identify_outliers_iqr,
    identify_outliers_zscore,
    setup_style,
    plot_distribution,
    plot_correlation_matrix,
    plot_categorical_comparison,
    generate_user_map
)

# Setup Visuals
setup_style()


## 1. Load Data
We load the **optimized Parquet files** from `data/processed` using `pyarrow` engine.

In [ ]:
df_details = load_data("details")
df_stats = load_data("stats")
df_profiles = load_data("profiles")
# Merge important details into stats for correlation analysis
# We likely want score, favorites, members from stats, and type, episodes from details


### 1.1 Data Inspection: Details


In [ ]:
print('Details Shape:', df_details.shape)
print('Details Dtypes:')
print(df_details.dtypes)
print('Details Head:')
display(df_details.head())
print('Details Missing Values:')
display(df_details.isna().sum())


### 1.2 Data Inspection: Stats


In [ ]:
print('Stats Shape:', df_stats.shape)
print('Stats Dtypes:')
print(df_stats.dtypes)
print('Stats Head:')
display(df_stats.head())
print('Stats Missing Values:')
display(df_stats.isna().sum())


### 1.3 Data Inspection: Profiles


In [ ]:
print('Profiles Shape:', df_profiles.shape)
print('Profiles Dtypes:')
print(df_profiles.dtypes)
print('Profiles Head:')
display(df_profiles.head())
print('Profiles Missing Values:')
display(df_profiles.isna().sum())


### 1.4 Merge Data


In [ ]:
df_merged = df_details[
    ['anime_id', 'title', 'type', 'source', 'episodes', 'rating', 'score', 'scored_by', 'rank', 'popularity', 'members', 'favorites']
].merge(
    df_stats[['anime_id', 'watching', 'completed']], 
    on='anime_id',
    how='inner'
)
print(f"Merged Dataset Shape: {df_merged.shape}")
print(f"Merged Dataset Shape: {df_merged.shape}")


## 2. Univariate Analysis
### A. Score Distribution (Normality)
Does the scoring follow a Bell Curve? Or is there a bias?

In [ ]:
plot_distribution(df_merged, 'score', title="Anime Score Distribution (Density)")

### B. Popularity (The Long Tail)
Checking metrics like `members` or `scored_by`. We expect a Power Law (Zipfian) distribution.
We check this by applying a Log Scale.

In [ ]:
plot_distribution(df_merged, 'members', log_scale=True, title="Members Distribution (Log Scale)")
plot_distribution(df_merged, 'favorites', log_scale=True, title="Favorites Distribution (Log Scale)")

## 3. Bivariate Analysis
### A. Correlation Matrix
Does popularity (`members`) correlate with quality (`score`)?

In [ ]:
cols_to_corr = ['score', 'scored_by', 'members', 'favorites', 'episodes']
plot_correlation_matrix(df_merged, columns=cols_to_corr)

### B. Categorical Comparisons
Do 'Movies' perform better than 'TV' series? Does source material matter?

In [ ]:
plot_categorical_comparison(df_merged, 'type', 'score')
plot_categorical_comparison(df_merged, 'source', 'score', top_n=10)

## 4. Outlier Detection
### A. Infinite Series (IQR on Episodes)
Identifying shows that run forever.

In [ ]:
outliers_ep, low, high = identify_outliers_iqr(df_merged, 'episodes')
print(f"\nTop 10 Longest Running Anime:")
display(outliers_ep[['title', 'episodes', 'type']].sort_values('episodes', ascending=False).head(10))

### B. Statistical Masterpieces (Z-Score > 3)
Anime distinctively superior to the rest.

In [ ]:
outliers_score = identify_outliers_zscore(df_merged, 'score', threshold=3.0)
print(f"\nTop Masterpieces (Z > 3):")
display(outliers_score[['title', 'score', 'rank', 'zscore']].sort_values('score', ascending=False))

## 5. Controversy Analysis
Using weighted standard deviation on vote counts to find polarizing shows.
High SD = Mixed Reception (Love/Hate).
Low SD = Consensus.

In [ ]:
# Requires full stats dataframe with vote columns
df_controversy = calculate_controversy_metric(df_stats)

# Merge title back for display
df_controversy = df_controversy.merge(
    df_details[['anime_id', 'title', 'scored_by', 'score']], 
    on='anime_id', 
    how='inner'
)

# Filter for relevant number of votes to avoid noise
print("Most Controversial Anime (High Vote Variance):")
complex_view = df_controversy[df_controversy['scored_by'] > 1000].sort_values('controversy_std', ascending=False)
display(complex_view[['title', 'score', 'controversy_std', 'scored_by']].head(10))

## 6. Geospatial Analysis
Where are the users coming from?

In [ ]:
generate_user_map(df_profiles)